# Collect and score autocorrelated additive structural causal process results

Run this notebook after any or all method notebooks in this directory. It
validates dataset and native-output hashes, converts each package's native
graph orientation in one shared tested module, and writes run-level and
grouped summary CSV files. Missing and failed fits remain visible.

The summary reports the mean and sample standard deviation across independent
dataset instances in each experimental cell. Exact-lag and aggregate-any-lag
recovery are separate estimands; aggregate-only methods are never assigned a
fabricated exact lag. AUROC and AUPRC are reported only when a method supplies
usable continuous scores and both edge classes are present.

In [ ]:
from __future__ import annotations

import platform
import sys
from pathlib import Path

EXPECTED_PYTHON = (3, 14)
if sys.version_info[:2] != EXPECTED_PYTHON:
    raise RuntimeError(
        "This notebook requires CPython 3.14; "
        f"found {platform.python_version()}."
    )

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "src" / "fastCGC" / "__init__.py").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the fastC-GC project root.")
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from fastCGC.experiments.comparison_scoring import score_dataset_outputs

In [ ]:
DATASET_KEY = "autocorrelatedSCM"
METHODS = (
    "fast-cgc",
    "fast-cgc-star",
    "cgc",
    "cgc-star",
    "pcmci-plus",
    "bagged-pcmci-plus",
    "cuts-plus",
)

# Leave both values as None to reproduce the archived decisions. After the
# independent calibration arm is complete, set both values (for example,
# DECISION_ALPHA = 0.01 and DECISION_BETA = 0.005) to write a separate
# rethresholded score directory. New fast-method archives must contain both
# component p-value arrays.
DECISION_ALPHA: float | None = None
DECISION_BETA: float | None = None

# Collection is safe to rerun. It reads immutable bundles and completed native
# outputs, records absent/failed fits explicitly, and atomically replaces only
# the two derived score tables.
artifacts = score_dataset_outputs(
    PROJECT_ROOT,
    DATASET_KEY,
    methods=METHODS,
    decision_alpha=DECISION_ALPHA,
    decision_beta=DECISION_BETA,
)
run_metrics = artifacts.run_metrics
summary = artifacts.summary

print(f"Run-level rows: {len(run_metrics)}")
print(run_metrics["status"].value_counts(dropna=False).to_string())
print(f"Run-level metrics: {artifacts.run_metrics_path}")
print(f"Mean (SD) summary: {artifacts.summary_path}")

In [ ]:
mean_std_columns = [
    "auprc_mean_std",
    "auroc_mean_std",
    "f1_mean_std",
    "mcc_mean_std",
    "normalized_shd_mean_std",
    "failure_aware_f1_mean_std",
]
identifier_columns = [
    "dataset",
    "method",
    "estimand",
    "n_variables",
    "n_samples",
    "fit_order",
    "noise",
    "regime",
    "expected",
    "missing",
    "attempted",
    "completed",
    "failed",
    "success_rate",
]
summary[identifier_columns + mean_std_columns]